# Evaluate RAL-CLIP by Streaming Target Images

Notebook này dùng **source real local-token cache** đã preprocess sẵn, rồi đọc target images trực tiếp để evaluate RAL-CLIP. Target local tokens không được lưu ra disk; chúng chỉ sống trong RAM/GPU lúc scoring.

Input chính: folder shard `.pt` như `ral_clip_local_tokens/` chứa `global`, `local`, `labels`, `paths`, `metadata`.

## Kaggle Setup

In [ ]:
# Chạy trên Kaggle nếu cần.
# !git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
# %cd /kaggle/working/training-free-tta-for-deepfake-detection
# !pip install -q -e . --no-deps
# !pip install -q open_clip_torch

## Imports

In [ ]:
from pathlib import Path
import math
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.mixture import GaussianMixture
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

repo_root = Path.cwd()
if (repo_root / 'code').exists():
    sys.path.insert(0, str(repo_root / 'code'))
else:
    sys.path.insert(0, str(repo_root))

from training.ffpp_split_utils import prepare_ffpp_split_dataframe


## Config

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42

CSV_PATH = Path('/kaggle/input/datasets/jamestashvik/deepfakebench/deepfakebench_dataset.csv')
DEEPFAKEBENCH_ROOT = Path('/kaggle/input/datasets/jamestashvik/deepfakebench/DeepFakeBench')

# Source real local-token shards. Nếu attach Kaggle dataset, đổi path này.
SOURCE_TOKEN_DIR = Path('/kaggle/input/ral-clip-local-tokens/ral_clip_local_tokens')
SOURCE_SHARD_GLOB = '*_shard*.pt'

# Target images: clean thì TARGET_REPLACEMENT_ROOT=None. Corruption thì trỏ tới root đã xử lý.
TARGET_DATASET_NAME = 'Celeb-DF-v1'      # 'Celeb-DF-v1' hoặc 'FaceForensics++'
TARGET_SPLIT_NAME = None                # Với FF++ dùng 'test'; CelebDF để None.
SPLIT_ROOT = None
TARGET_REPLACEMENT_ROOT = None          # Ví dụ: Path('/kaggle/input/.../processed_output/color_contrast/level_1/Celeb-DF-v1')
TARGET_NAME = 'celebdfv1-clean'

# Optional quick/debug cap. None = full target.
MAX_TARGET = None

RETRIEVE_M = 16
TOP_K_PATCHES = 16
NEIGHBOR_RADIUS = 1
BATCH_SIZE = 16
NUM_WORKERS = 2
USE_AMP = True

# Nếu source cache dùng stride/center, notebook sẽ đọc metadata và áp dụng y chang cho target.
OUTPUT_DIR = Path('/kaggle/working/ral_clip_stream_eval')
SCORES_OUTPUT = OUTPUT_DIR / f'{TARGET_NAME}_ral_clip_stream_scores.csv'
SUMMARY_OUTPUT = OUTPUT_DIR / f'{TARGET_NAME}_ral_clip_stream_summary.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
print('device:', DEVICE)

## Load Source Real Memory

In [ ]:
def load_source_memory(source_dir, pattern):
    paths = sorted(Path(source_dir).glob(pattern))
    if not paths:
        raise FileNotFoundError(f'No source token shards found: {source_dir}/{pattern}')
    globals_, labels_, all_paths = [], [], []
    locals_by_key = None
    metadata = None
    for path in paths:
        payload = torch.load(path, map_location='cpu')
        if metadata is None:
            metadata = payload.get('metadata', {})
            locals_by_key = {key: [] for key in payload['local']}
        globals_.append(payload['global'].float())
        labels_.append(payload['labels'].long())
        all_paths.extend(payload.get('paths', []))
        for key, value in payload['local'].items():
            locals_by_key[key].append(value.float())
    labels = torch.cat(labels_).long()
    counts = torch.bincount(labels, minlength=2).tolist()
    print('source shards:', len(paths))
    print('source label counts [REAL, FAKE]:', counts)
    if counts[1] > 0:
        print('WARNING: source cache contains FAKE too. Filtering to REAL only for RAL-CLIP memory.')
    real_idx = torch.where(labels.eq(0))[0]
    memory = {
        'global': F.normalize(torch.cat(globals_).float(), dim=-1)[real_idx].contiguous(),
        'local': {key: F.normalize(torch.cat(parts).float(), dim=-1)[real_idx].contiguous() for key, parts in locals_by_key.items()},
        'labels': labels[real_idx].contiguous(),
        'paths': [all_paths[int(i)] for i in real_idx] if all_paths else [],
        'metadata': metadata or {},
    }
    print('source real memory:', tuple(memory['global'].shape))
    for key, value in memory['local'].items():
        print(key, tuple(value.shape))
    print('metadata:', memory['metadata'])
    return memory

source_memory = load_source_memory(SOURCE_TOKEN_DIR, SOURCE_SHARD_GLOB)
SOURCE_METADATA = source_memory['metadata']
CLIP_MODEL = SOURCE_METADATA.get('clip_model', 'ViT-L-14/openai').split('/')[0]
PRETRAINED = SOURCE_METADATA.get('clip_model', 'ViT-L-14/openai').split('/')[1]
LAYERS = SOURCE_METADATA.get('layers', [-6])
PATCH_KEEP_MODE = SOURCE_METADATA.get('patch_keep_mode', 'all')
PATCH_STRIDE = int(SOURCE_METADATA.get('patch_stride', 1) or 1)
PATCH_CENTER_FRACTION = float(SOURCE_METADATA.get('patch_center_fraction', 1.0) or 1.0)
print('using target extraction config:', CLIP_MODEL, PRETRAINED, LAYERS, PATCH_KEEP_MODE, PATCH_STRIDE, PATCH_CENTER_FRACTION)

## Prepare Target DataFrame

In [ ]:
def find_dataset_root(root, dataset_name):
    root = Path(root)
    candidates = [root / dataset_name, root]
    if dataset_name == 'Celeb-DF-v1':
        anchors = ('Celeb-real', 'YouTube-real', 'Celeb-synthesis')
    else:
        anchors = ('original_sequences', 'manipulated_sequences')
    for candidate in candidates:
        if any((candidate / anchor).exists() for anchor in anchors):
            return candidate
    return root

def prepare_generic_dataframe(csv_path, dataset_name, deepfakebench_root, replacement_root=None):
    df = pd.read_csv(csv_path)
    df = df[df['datasetname'].eq(dataset_name)].copy()
    if df.empty:
        raise ValueError(f'No rows found for datasetname={dataset_name!r}')
    df['label_num'] = df['label'].map({'REAL': 0, 'FAKE': 1}).astype(int)
    df['imagepath_fixed'] = df['imagepath'].astype(str).str.replace('../input/deepfakebench', str(deepfakebench_root), regex=False)
    if replacement_root is not None:
        source_root = Path(deepfakebench_root) / dataset_name
        target_root = find_dataset_root(replacement_root, dataset_name)
        df['imagepath_fixed'] = df['imagepath_fixed'].astype(str).str.replace(str(source_root), str(target_root), regex=False)
        print('target replacement root:', target_root)
    exists = df['imagepath_fixed'].map(lambda p: Path(p).exists())
    missing = int((~exists).sum())
    if missing:
        print(f'dropping missing target files: {missing}/{len(df)}')
        print(df.loc[~exists, ['imagepath', 'imagepath_fixed']].head(10).to_string(index=False))
        df = df[exists].copy()
    return df.reset_index(drop=True)

if TARGET_DATASET_NAME == 'FaceForensics++' and TARGET_SPLIT_NAME:
    target_df = prepare_ffpp_split_dataframe(CSV_PATH, DEEPFAKEBENCH_ROOT, TARGET_SPLIT_NAME, split_root=SPLIT_ROOT)
    if TARGET_REPLACEMENT_ROOT is not None:
        source_root = Path(DEEPFAKEBENCH_ROOT) / 'FaceForensics++'
        target_root = find_dataset_root(TARGET_REPLACEMENT_ROOT, 'FaceForensics++')
        target_df['imagepath_fixed'] = target_df['imagepath_fixed'].astype(str).str.replace(str(source_root), str(target_root), regex=False)
        print('target replacement root:', target_root)
else:
    target_df = prepare_generic_dataframe(CSV_PATH, TARGET_DATASET_NAME, DEEPFAKEBENCH_ROOT, TARGET_REPLACEMENT_ROOT)

if MAX_TARGET is not None and len(target_df) > MAX_TARGET:
    target_df = target_df.sample(n=MAX_TARGET, random_state=SEED).reset_index(drop=True)

print('target rows:', len(target_df))
print(target_df['label_num'].value_counts().rename(index={0: 'REAL', 1: 'FAKE'}))
display(target_df[['label', 'label_num', 'imagepath_fixed']].head())

## CLIP Extractor

In [ ]:
class TargetImageDataset(Dataset):
    def __init__(self, dataframe, preprocess):
        self.df = dataframe.reset_index(drop=True)
        self.preprocess = preprocess
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['imagepath_fixed']).convert('RGB')
        return self.preprocess(image), int(row['label_num']), row['imagepath_fixed']

def patch_keep_indices(num_patches):
    side = int(round(math.sqrt(num_patches)))
    if side * side != num_patches or PATCH_KEEP_MODE == 'all':
        return torch.arange(num_patches, dtype=torch.long)
    if PATCH_KEEP_MODE == 'stride':
        idx = [r * side + c for r in range(0, side, PATCH_STRIDE) for c in range(0, side, PATCH_STRIDE)]
        return torch.tensor(idx, dtype=torch.long)
    if PATCH_KEEP_MODE == 'center':
        keep_side = max(1, int(round(side * PATCH_CENTER_FRACTION)))
        start = max(0, (side - keep_side) // 2)
        end = min(side, start + keep_side)
        idx = [r * side + c for r in range(start, end) for c in range(start, end)]
        return torch.tensor(idx, dtype=torch.long)
    raise ValueError(f'Unknown PATCH_KEEP_MODE={PATCH_KEEP_MODE}')

class OpenClipLocalExtractor:
    def __init__(self, model, layers):
        self.model = model.eval()
        self.layers = list(layers)
        self.captures = {}
        self.handles = []
        blocks = self._visual_blocks()
        n_blocks = len(blocks)
        self.layer_indices = [layer if layer >= 0 else n_blocks + layer for layer in self.layers]
        for idx in self.layer_indices:
            self.handles.append(blocks[idx].register_forward_hook(self._make_hook(idx)))
        print('hooked layers:', self.layer_indices)
    def _visual_blocks(self):
        visual = self.model.visual
        if hasattr(visual, 'transformer') and hasattr(visual.transformer, 'resblocks'):
            return visual.transformer.resblocks
        if hasattr(visual, 'trunk') and hasattr(visual.trunk, 'blocks'):
            return visual.trunk.blocks
        raise TypeError('Cannot find visual transformer blocks')
    def _make_hook(self, idx):
        def hook(module, inputs, output):
            self.captures[idx] = (output[0] if isinstance(output, tuple) else output).detach()
        return hook
    def close(self):
        for handle in self.handles:
            handle.remove()
    @torch.inference_mode()
    def __call__(self, images):
        self.captures = {}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(USE_AMP and images.device.type == 'cuda')):
            global_features = self.model.encode_image(images)
        global_features = F.normalize(global_features.float(), dim=-1).cpu()
        locals_by_key = {}
        batch_size = images.shape[0]
        for raw_layer, idx in zip(self.layers, self.layer_indices):
            tokens = self.captures[idx].float()
            if tokens.shape[1] == batch_size:
                tokens = tokens.permute(1, 0, 2).contiguous()
            patch_tokens = F.normalize(tokens[:, 1:, :], dim=-1)
            keep_idx = patch_keep_indices(patch_tokens.shape[1])
            locals_by_key[f'layer_{raw_layer}'] = patch_tokens[:, keep_idx, :].cpu().contiguous()
        return global_features, locals_by_key

import open_clip
clip_model, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=PRETRAINED, device=DEVICE)
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad_(False)
extractor = OpenClipLocalExtractor(clip_model, LAYERS)

## RAL-CLIP Scoring

In [ ]:
def local_deviation_one(target_z, real_z, neighbor_radius=1):
    n = target_z.shape[0]
    side = int(round(math.sqrt(n)))
    if side * side != n:
        neighbor_radius = 0
    out = []
    for i in range(n):
        if neighbor_radius > 0:
            r, c = divmod(i, side)
            js = []
            for rr in range(max(0, r - neighbor_radius), min(side, r + neighbor_radius + 1)):
                for cc in range(max(0, c - neighbor_radius), min(side, c + neighbor_radius + 1)):
                    js.append(rr * side + cc)
            candidates = real_z[:, js, :].reshape(-1, real_z.shape[-1])
        else:
            candidates = real_z[:, i, :]
        out.append(1.0 - torch.matmul(candidates, target_z[i]).max())
    return torch.stack(out)

def score_batch(target_global, target_local, memory, retrieve_m=16, top_k=16, neighbor_radius=1):
    sims = target_global @ memory['global'].T
    retrieve_m = min(retrieve_m, memory['global'].shape[0])
    nn_idx = sims.topk(k=retrieve_m, dim=1).indices
    scores = []
    layer_keys = list(target_local.keys())
    for row in range(target_global.shape[0]):
        layer_scores = []
        idx = nn_idx[row]
        for key in layer_keys:
            patch_d = local_deviation_one(target_local[key][row], memory['local'][key][idx], neighbor_radius=neighbor_radius)
            k = min(top_k, patch_d.numel())
            layer_scores.append(float(patch_d.topk(k).values.mean()))
        scores.append(float(np.mean(layer_scores)))
    return scores

def gmm_threshold(scores):
    scores = np.asarray(scores, dtype=np.float64)
    gmm = GaussianMixture(n_components=2, random_state=SEED)
    gmm.fit(scores.reshape(-1, 1))
    means = gmm.means_.ravel()
    order = np.argsort(means)
    lo, hi = means[order[0]], means[order[1]]
    grid = np.linspace(scores.min(), scores.max(), 4096)
    logprob = gmm._estimate_weighted_log_prob(grid.reshape(-1, 1))
    diff = logprob[:, order[0]] - logprob[:, order[1]]
    between = (grid >= lo) & (grid <= hi)
    if between.any():
        ids = np.where(between)[0]
        return float(grid[ids[np.argmin(np.abs(diff[ids]))]])
    return float((lo + hi) / 2)

def calculate_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[idx] + fnr[idx]) / 2), float(thresholds[idx])


## Run Evaluation

In [ ]:
loader = DataLoader(TargetImageDataset(target_df, preprocess), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

rows = []
for images, labels, paths in tqdm(loader):
    images = images.to(DEVICE, non_blocking=True)
    target_global, target_local = extractor(images)
    scores = score_batch(target_global, target_local, source_memory, retrieve_m=RETRIEVE_M, top_k=TOP_K_PATCHES, neighbor_radius=NEIGHBOR_RADIUS)
    for path, label, score in zip(paths, labels.tolist(), scores):
        rows.append({'dataset': TARGET_NAME, 'path': path, 'label': int(label), 'score': float(score)})

scores_df = pd.DataFrame(rows)
threshold = gmm_threshold(scores_df['score'].values)
scores_df['pred'] = (scores_df['score'] > threshold).astype(int)
scores_df['threshold'] = threshold
scores_df.to_csv(SCORES_OUTPUT, index=False)
print('saved scores:', SCORES_OUTPUT)

y_true = scores_df['label'].values
y_score = scores_df['score'].values
y_pred = scores_df['pred'].values
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
eer, eer_threshold = calculate_eer(y_true, y_score)
summary = pd.DataFrame([{
    'dataset': TARGET_NAME,
    'method': 'ral_clip_stream_local',
    'source_token_dir': str(SOURCE_TOKEN_DIR),
    'target_dataset_name': TARGET_DATASET_NAME,
    'target_replacement_root': None if TARGET_REPLACEMENT_ROOT is None else str(TARGET_REPLACEMENT_ROOT),
    'retrieve_m': RETRIEVE_M,
    'top_k_patches': TOP_K_PATCHES,
    'neighbor_radius': NEIGHBOR_RADIUS,
    'threshold': threshold,
    'acc': accuracy_score(y_true, y_pred),
    'f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
    'auc': roc_auc_score(y_true, y_score),
    'ap': average_precision_score(y_true, y_score),
    'eer': eer,
    'eer_threshold': eer_threshold,
    'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
}])
summary.to_csv(SUMMARY_OUTPUT, index=False)
print('saved summary:', SUMMARY_OUTPUT)
display(summary)
display(scores_df.head())
extractor.close()